# Задание 2 — Информационный поиск (CRANFIELD)

Для запроса `interference effects.` найдём 2 ближайших документа:
- по коэффициенту Жаккара,
- по косинусному расстоянию.

In [2]:
import os
import tarfile
import urllib.request

import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer


def load_cran_queries(data_dir: str = "./data"):
    os.makedirs(data_dir, exist_ok=True)
    archive_path = os.path.join(data_dir, "cran.tar.gz")
    extract_path = os.path.join(data_dir, "cran")
    qry_path = os.path.join(extract_path, "cran.qry")

    if not os.path.exists(qry_path):
        urllib.request.urlretrieve(
            "http://ir.dcs.gla.ac.uk/resources/test_collections/cran/cran.tar.gz",
            archive_path,
        )
        with tarfile.open(archive_path) as archive:
            archive.extractall(extract_path)

    raw_query_data = [
        line.strip()
        for line in open(qry_path, "r", encoding="utf-8", errors="ignore")
        if not line.startswith(".")
    ]

    query_data = [""]
    for query_part in raw_query_data:
        query_data[-1] += query_part + " "
        if query_part.endswith("."):
            query_data.append("")

    if query_data and not query_data[-1].strip():
        query_data = query_data[:-1]

    return query_data


query_data = load_cran_queries()
len(query_data)

/var/folders/_5/267hyv_x24l2v01mjkhyfkgc0000gn/T/ipykernel_27091/1862274079.py:21: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  archive.extractall(extract_path)


226

In [3]:
def jaccard_sim(vector_a: np.ndarray, vector_b: np.ndarray) -> float:
    intersection = np.logical_and(vector_a, vector_b).sum()
    union = np.logical_or(vector_a, vector_b).sum()
    return intersection / union if union != 0 else 0.0


def cosine_distance(vector_a: np.ndarray, vector_b: np.ndarray) -> float:
    norm_a = np.linalg.norm(vector_a)
    norm_b = np.linalg.norm(vector_b)
    if norm_a == 0 or norm_b == 0:
        return 1.0
    return 1 - np.dot(vector_a, vector_b) / (norm_a * norm_b)


QUERY = ["interference effects."]

encoder = CountVectorizer(binary=True)
encoded_data = encoder.fit_transform(query_data)
encoded_query = encoder.transform(QUERY)[0].todense().A1
docs_bin = [doc.todense().A1 for doc in encoded_data]

jaccard_scores = [
    (doc_id, jaccard_sim(encoded_query, doc))
    for doc_id, doc in enumerate(docs_bin)
]
jaccard_scores = sorted(jaccard_scores, key=lambda x: x[1], reverse=True)

tfidf_encoder = TfidfVectorizer()
tfidf_data = tfidf_encoder.fit_transform(query_data)
tfidf_query = tfidf_encoder.transform(QUERY)[0].todense().A1
docs_tfidf = [doc.todense().A1 for doc in tfidf_data]

cosine_scores = [
    (doc_id, cosine_distance(tfidf_query, doc))
    for doc_id, doc in enumerate(docs_tfidf)
]
cosine_scores = sorted(cosine_scores, key=lambda x: x[1])

print("Top-2 Jaccard:")
for i, score in jaccard_scores[:2]:
    print(i, round(score, 2), query_data[i])

print("\nTop-2 Cosine distance:")
for i, score in cosine_scores[:2]:
    print(i, round(score, 2), query_data[i])

Top-2 Jaccard:
90 0.25 what interference effects are likely at transonic speeds . 
33 0.2 have wind tunnel interference effects been investigated on a systematic basis . 

Top-2 Cosine distance:
90 0.5 what interference effects are likely at transonic speeds . 
33 0.59 have wind tunnel interference effects been investigated on a systematic basis . 


In [4]:
j1_idx, j1_val = jaccard_scores[0][0], round(jaccard_scores[0][1], 2)
j2_idx, j2_val = jaccard_scores[1][0], round(jaccard_scores[1][1], 2)

c1_idx, c1_val = cosine_scores[0][0], round(cosine_scores[0][1], 2)
c2_idx, c2_val = cosine_scores[1][0], round(cosine_scores[1][1], 2)

print("Ответы для формы:")
print("Jaccard #1 index:", j1_idx)
print("Jaccard #1 value:", j1_val)
print("Jaccard #2 index:", j2_idx)
print("Jaccard #2 value:", j2_val)
print("Cosine #1 index:", c1_idx)
print("Cosine #1 distance:", c1_val)
print("Cosine #2 index:", c2_idx)
print("Cosine #2 distance:", c2_val)

Ответы для формы:
Jaccard #1 index: 90
Jaccard #1 value: 0.25
Jaccard #2 index: 33
Jaccard #2 value: 0.2
Cosine #1 index: 90
Cosine #1 distance: 0.5
Cosine #2 index: 33
Cosine #2 distance: 0.59
